# Phase 2A TTS spike — clone "A Hít Official" voice (Colab, free GPU)

Runs **F5-TTS Vietnamese** (fast, cc-by-nc) and optionally **GPT-SoVITS** (MIT, preferred for a monetized channel).

**Before running:** Runtime → Change runtime type → **T4 GPU**.

**You need:** `sample.wav` from the repo (`assets/voice/sample.wav`, ~7.7s clean clip) and its transcript (`assets/voice/sample.txt`).

At the end you get: a synthesized `.wav`, the wall-time + real-time-factor, and the exact command line to paste into `scripts/tts_f5.sh` / `scripts/tts_gptsovits.sh`.

## 1. Check GPU + upload the reference clip

In [ ]:
!nvidia-smi -L
from google.colab import files
print('\nUpload assets/voice/sample.wav (the ~7.7s reference clip):')
up = files.upload()
REF_WAV = next(iter(up))
print('ref wav =', REF_WAV)

REF_TEXT = "Chúng ta cùng nhau xây dựng một cộng đồng AI automation. Xin chào và hẹn gặp lại các bạn ở những video tiếp theo."
# If your sample.txt differs, paste it here instead:
# REF_TEXT = open('sample.txt', encoding='utf-8').read().strip()

GEN_TEXT = (
    "Hôm nay AI lại vừa có một bước nhảy lớn. "
    "Một mô hình mới vừa được công bố, nhanh gấp đôi bản trước và chi phí giảm gần một nửa. "
    "Nghĩa là người bình thường cũng chạy được những tác vụ trước đây rất tốn kém. "
    "Các công ty lớn đã tích hợp nó rồi, còn bạn vẫn đang làm thủ công. "
    "Khoảng cách đang giãn ra mỗi ngày."
)
print('\ngen text (~40s target):\n', GEN_TEXT)

## 2. F5-TTS Vietnamese (cc-by-nc — fallback engine)

In [ ]:
!pip -q install f5-tts huggingface_hub soundfile
from huggingface_hub import hf_hub_download
# Better quality (5.4GB): hynt/F5-TTS-Vietnamese-ViVoice / model_last.pt
# Lighter (1.35GB):       yukiakai/F5-TTS-Vietnamese / model_85044.safetensors
CKPT = hf_hub_download('hynt/F5-TTS-Vietnamese-ViVoice', 'model_last.pt')
try:
    VOCAB = hf_hub_download('hynt/F5-TTS-Vietnamese-ViVoice', 'vocab.txt')
except Exception:
    VOCAB = hf_hub_download('yukiakai/F5-TTS-Vietnamese', 'vocab.txt')
print('ckpt =', CKPT, '\nvocab =', VOCAB)

In [ ]:
import time, subprocess, soundfile as sf, os
os.makedirs('out', exist_ok=True)
cmd = [
    'f5-tts_infer-cli', '-m', 'F5TTS_v1_Base', '-p', CKPT, '-v', VOCAB,
    '-r', REF_WAV, '-s', REF_TEXT, '-t', GEN_TEXT,
    '-o', 'out', '-w', 'f5_vi.wav', '--nfe_step', '32',
]
t0 = time.time()
print(subprocess.run(cmd, capture_output=True, text=True).stderr[-2000:])
wall = time.time() - t0
audio_s = sf.info('out/f5_vi.wav').duration
print(f'\nWALL {wall:.1f}s  |  AUDIO {audio_s:.1f}s  |  RTF {wall/audio_s:.2f}  (GPU)')

In [ ]:
from IPython.display import Audio, display
print('Reference:'); display(Audio(REF_WAV))
print('F5-TTS Vietnamese clone:'); display(Audio('out/f5_vi.wav'))

## 3. (optional) GPT-SoVITS — MIT license, preferred for production

Heavier setup. Run only if F5-TTS quality is borderline and you want the MIT-licensed path.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS
%cd GPT-SoVITS
!pip -q install -r requirements.txt
# Pretrained models (GPT-SoVITS v2):
!huggingface-cli download lj1995/GPT-SoVITS --local-dir GPT_SoVITS/pretrained_models
import time, subprocess, soundfile as sf
# GPT-SoVITS zero-shot via its API script; flag names vary by revision —
# confirm against GPT_SoVITS/inference_cli.py --help, then copy the working
# invocation into scripts/tts_gptsovits.sh.
cmd = ['python', 'GPT_SoVITS/inference_cli.py',
       '--ref_audio', f'/content/{REF_WAV}', '--ref_text', REF_TEXT, '--ref_language', 'vi',
       '--target_text', GEN_TEXT, '--target_language', 'vi', '--output', '/content/out/gptsovits.wav']
t0 = time.time()
print(subprocess.run(cmd, capture_output=True, text=True).stderr[-3000:])
wall = time.time() - t0
try:
    print(f'WALL {wall:.1f}s  |  RTF {wall/sf.info("/content/out/gptsovits.wav").duration:.2f}')
    from IPython.display import Audio, display; display(Audio('/content/out/gptsovits.wav'))
except Exception as e:
    print('no output:', e)

## 4. Download results + fill the spike note

In [ ]:
from google.colab import files
for p in ['out/f5_vi.wav', '/content/out/gptsovits.wav']:
    try: files.download(p)
    except Exception: pass
print('Send the wav(s) back + paste the WALL/RTF numbers and the working command into')
print('docs/superpowers/notes/2026-09-03-tts-spike.md, then set config/settings.yaml video.tts_provider.')